In [1]:
# -*- coding: utf-8 -*-
"""
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 1: FACTORES INTERNOS (CORE DEL ESTRES)
================================================================================
TFM: Analisis multidimensional del estres estudiantil mediante Business
     Analytics e Inteligencia Artificial

Datasets analizados (4):
  1.1 academic Stress level - maintainance 1.csv  (Kaggle: poushal02)
  1.2 anxiety.csv                                  (Kaggle: petalme)
  1.3 MentalHealthSurvey.csv                       (Kaggle: abdullahashfaqvirk)
  1.4 student_habits_performance.csv               (GitHub: abhi88377)

Metodologia aplicada a cada dataset (5 pasos):
  PASO 1 - Carga e inventario: dimensiones, tipos de variables
  PASO 2 - Calidad de datos: nulos, duplicados, valores anomalos
  PASO 3 - Analisis univariante: distribucion de la variable objetivo
  PASO 4 - Analisis bivariante: correlaciones / crosstabs con el estres
  PASO 5 - Hallazgos: sintesis de 3-5 conclusiones del dataset
================================================================================
"""
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Configuracion global ──────────────────────────────────────────────────────
import os

BASE = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta"
DATA = os.path.join(BASE, "Bloque 1") + os.sep
FIG  = os.path.join(BASE, "figuras1") + os.sep
os.makedirs(FIG, exist_ok=True)   # crea la carpeta de figuras1 si no existe

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

# ==============================================================================
# DATASET 1.1 — ACADEMIC STRESS (poushal02, n=140)
# ==============================================================================
print(SEP); print("DATASET 1.1 - ACADEMIC STRESS (poushal02)"); print(SEP)

# PASO 1: Carga e inventario -----------------------------------------------
df1 = pd.read_csv(DATA + "1 Student Academic Stress Real World Dataset/"
                         "academic Stress level - maintainance 1.csv")
df1.columns = [c.strip() for c in df1.columns]          # limpieza de espacios
TARGET1 = "Rate your academic stress index"
print(f"Dimensiones: {df1.shape[0]} filas x {df1.shape[1]} columnas")
print("Variables:", list(df1.columns))

# PASO 2: Calidad ------------------------------------------------------------
print("\nNulos por columna:\n", df1.isna().sum()[df1.isna().sum() > 0])
print("Duplicados:", df1.duplicated().sum())

# PASO 3: Univariante --------------------------------------------------------
dist1 = df1[TARGET1].value_counts().sort_index()
print("\nDistribucion del indice de estres (1-5):\n", dist1)
pct_alto = (df1[TARGET1] >= 4).mean() * 100
print(f"% de estudiantes en niveles 4-5: {pct_alto:.1f}%")

# PASO 4: Bivariante ---------------------------------------------------------
print("\nEstres medio por etapa academica:")
print(df1.groupby("Your Academic Stage")[TARGET1].agg(["mean", "count"]).round(2))
print("\nEstres medio segun presion de pares (peer pressure):")
print(df1.groupby("Peer pressure")[TARGET1].mean().round(2))

# Figura 1.1
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.countplot(data=df1, x=TARGET1, ax=axes[0], color="#1F5C99")
axes[0].set_title(f"Distribución del índice de estrés académico (n={len(df1)})")
axes[0].set_xlabel("Índice de estrés (1=bajo, 5=alto)")
axes[0].set_ylabel("Estudiantes")
sns.boxplot(data=df1, x="Your Academic Stage", y=TARGET1, ax=axes[1],
            hue="Your Academic Stage", palette="Blues", legend=False)
axes[1].set_title("Estrés por etapa académica")
axes[1].set_xlabel(""); axes[1].set_ylabel("Índice de estrés")
plt.tight_layout()
plt.savefig(FIG + "b1_1_academic_stress.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 1.1:
 - Mas del 60% de la muestra reporta estres en niveles 4-5.
 - La etapa de grado (undergraduate) concentra la mayoria de la muestra (100/140).
 - La presion de pares se asocia a mayor indice de estres.
 - Dataset pequeno (n=140): util para EDA descriptivo, limitado para modelar.
""")

# ==============================================================================
# DATASET 1.2 — ANXIETY / GAD-7 (petalme, n=13.464)
# ==============================================================================
print(SEP); print("DATASET 1.2 - ANXIETY GAD-7 (petalme)"); print(SEP)

# PASO 1: Carga (encoding latin1: el archivo no es UTF-8) --------------------
df2 = pd.read_csv(DATA + "1 Student anxiety dataset/anxiety.csv",
                  encoding="latin1")
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")
print("Escalas incluidas: GAD-7 (ansiedad), SWL (satisfaccion vital), "
      "SPIN (ansiedad social)")

# PASO 2: Calidad + subconjunto estudiantil ----------------------------------
nulos2 = df2[["GAD_T", "SWL_T", "SPIN_T", "Hours", "Age"]].isna().sum()
print("\nNulos en variables clave:\n", nulos2)
students = df2[df2["Work"].isin(
    ["Student at college / university", "Student at school"])].copy()
print(f"Subconjunto estudiantes (variable Work): n={len(students)}")

# PASO 3: Univariante --------------------------------------------------------
print("\nGAD_T (0-21) describe:\n", students["GAD_T"].describe().round(2))
# Umbrales clinicos GAD-7: 5=leve, 10=moderada, 15=severa
severidad = pd.cut(students["GAD_T"], bins=[-1, 4, 9, 14, 21],
                   labels=["Mínima", "Leve", "Moderada", "Severa"])
print("\nSeveridad clinica (umbrales GAD-7):\n",
      severidad.value_counts(normalize=True).round(3) * 100)

# PASO 4: Bivariante ---------------------------------------------------------
corr2 = students[["GAD_T", "SWL_T", "SPIN_T", "Hours", "Age"]].corr().round(2)
print("\nMatriz de correlaciones:\n", corr2)

# Figura 1.2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(students["GAD_T"], bins=22, ax=axes[0], color="#1F5C99")
axes[0].set_title(f"Distribución GAD_T — ansiedad (estudiantes, n={len(students)})")
axes[0].set_xlabel("Puntuación GAD-7 (0-21)")
sns.scatterplot(data=students.sample(3000, random_state=42),
                x="SWL_T", y="GAD_T", alpha=0.2, ax=axes[1], color="#C44E52")
axes[1].set_title("Ansiedad vs Satisfacción con la vida (r = -0.40)")
axes[1].set_xlabel("SWL_T (satisfacción vital)"); axes[1].set_ylabel("GAD_T")
plt.tight_layout()
plt.savefig(FIG + "b1_2_anxiety_gad.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 1.2:
 - Correlacion GAD_T vs SWL_T = -0.40: mas ansiedad, menos satisfaccion vital.
 - Correlacion GAD_T vs SPIN_T = +0.44: la ansiedad general y la social co-ocurren.
 - Las horas de juego apenas correlacionan con la ansiedad (r=0.07).
 - Contexto original: jugadores online; el subset estudiantil (n=9.302) es valido
   para analizar ansiedad en jovenes, con esa salvedad documentada.
""")

# ==============================================================================
# DATASET 1.3 — MENTAL HEALTH SURVEY (abdullahashfaqvirk, n=87)
# ==============================================================================
print(SEP); print("DATASET 1.3 - MENTAL HEALTH SURVEY (abdullahashfaqvirk)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df3 = pd.read_csv(DATA + "1 Student Mental Health Survey/MentalHealthSurvey.csv")
print(f"Dimensiones: {df3.shape[0]} filas x {df3.shape[1]} columnas")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos totales:", df3.isna().sum().sum())
print("Nota: cgpa y average_sleep vienen como rangos de texto -> recodificar "
      "a ordinal antes del modelado (Objetivo 2).")

# PASO 3: Univariante --------------------------------------------------------
print("\nDistribucion depression (1-5):\n",
      df3["depression"].value_counts().sort_index())
print("Distribucion anxiety (1-5):\n",
      df3["anxiety"].value_counts().sort_index())

# PASO 4: Bivariante ---------------------------------------------------------
num3 = df3.select_dtypes("number")
corr3 = num3.corr()[["depression", "anxiety"]].round(2)
print("\nCorrelacion de cada variable con depression y anxiety:\n",
      corr3.sort_values("depression", ascending=False))

# Figura 1.3
plt.figure(figsize=(7.5, 6))
sns.heatmap(num3.corr(), annot=True, fmt=".2f", cmap="vlag", center=0,
            annot_kws={"size": 8})
plt.title(f"Matriz de correlaciones — Mental Health Survey (n={len(df3)})")
plt.tight_layout()
plt.savefig(FIG + "b1_3_mentalhealthsurvey_corr.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 1.3:
 - Aislamiento social: r=0.71 con depresion (el predictor mas fuerte del bloque).
 - Inseguridad ante el futuro: r=0.47 con depresion.
 - Carga academica: r=0.38 con depresion y ansiedad.
 - La satisfaccion academica actua como factor protector (r negativo).
 - n=87: correlaciones consistentes entre si a pesar del tamano reducido.
""")

# ==============================================================================
# DATASET 1.4 — STUDENT HABITS VS PERFORMANCE (abhi88377, n=1000)
# ==============================================================================
print(SEP); print("DATASET 1.4 - STUDENT HABITS VS PERFORMANCE (abhi88377)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df4 = pd.read_csv(DATA + "1 Student-Habits-vs-Academic-Performance/"
                         "student_habits_performance.csv")
print(f"Dimensiones: {df4.shape[0]} filas x {df4.shape[1]} columnas")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos:", dict(df4.isna().sum()[df4.isna().sum() > 0]))

# PASO 3: Univariante --------------------------------------------------------
print("\nmental_health_rating (1-10):\n",
      df4["mental_health_rating"].value_counts().sort_index())
print(">>> Distribucion casi uniforme: primer indicio de datos sinteticos.")

# PASO 4: Bivariante ---------------------------------------------------------
num4 = df4.select_dtypes("number")
print("\nCorrelacion con mental_health_rating:\n",
      num4.corr()["mental_health_rating"].round(2).sort_values())
print("\nCorrelacion con exam_score (validacion de coherencia interna):\n",
      num4.corr()["exam_score"].round(2).sort_values(ascending=False).head(4))

# Figura 1.4
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(df4["mental_health_rating"], bins=10, ax=axes[0], color="#55A868")
axes[0].set_title("mental_health_rating: distribución uniforme\n(indicio de datos sintéticos)")
sns.regplot(data=df4, x="study_hours_per_day", y="exam_score", ax=axes[1],
            scatter_kws={"alpha": 0.2}, line_kws={"color": "#C44E52"})
axes[1].set_title("Horas de estudio vs nota (r = 0.83)\núnica relación real del dataset")
plt.tight_layout()
plt.savefig(FIG + "b1_4_student_habits.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 1.4:
 - mental_health_rating NO correlaciona con ningun habito (r ~ 0.00 en todos).
 - La unica relacion fuerte es horas de estudio -> nota de examen (r=0.83).
 - CONCLUSION: dataset sintetico para la dimension de salud mental. Se usara
   para validar el pipeline tecnico del Objetivo 2, no para conclusiones.
 - 91 nulos (9.1%) en parental_education_level: imputar por moda o excluir.
""")

print(SEP)
print("EDA BLOQUE 1 COMPLETADO - 4 figuras generadas en", FIG)
print(SEP)

DATASET 1.1 - ACADEMIC STRESS (poushal02)
Dimensiones: 140 filas x 9 columnas
Variables: ['Timestamp', 'Your Academic Stage', 'Peer pressure', 'Academic pressure from your home', 'Study Environment', 'What coping strategy you use as a student?', 'Do you have any bad habits like smoking, drinking on a daily basis?', 'What would you rate the academic  competition in your student life', 'Rate your academic stress index']

Nulos por columna:
 Study Environment    1
dtype: int64
Duplicados: 0

Distribucion del indice de estres (1-5):
 Rate your academic stress index
1     6
2     9
3    36
4    56
5    33
Name: count, dtype: int64
% de estudiantes en niveles 4-5: 63.6%

Estres medio por etapa academica:
                     mean  count
Your Academic Stage             
high school          3.83     29
post-graduate        3.73     11
undergraduate        3.69    100

Estres medio segun presion de pares (peer pressure):
Peer pressure
1    2.62
2    3.29
3    3.75
4    4.06
5    4.57
Name: Rat

In [2]:
# -*- coding: utf-8 -*-
"""
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 2: FACTORES EXTERNOS (HABITOS DIGITALES)
================================================================================
Datasets analizados (3):
  2.1 social_media_sleep_stress_productivity_11000.csv  (Kaggle: jayjoshi37)
  2.2 smmh.csv                                           (Kaggle: souvikahmed071)
  2.3 Mental_Health_Lifestyle_Dataset.csv                (GitHub: datafit17)

Metodologia (misma plantilla de 5 pasos que el Bloque 1).
================================================================================
"""
# ── Configuracion global ──────────────────────────────────────────────────────

DATA = os.path.join(BASE, "Bloque 2") + os.sep

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

# ==============================================================================
# DATASET 2.1 — SOCIAL MEDIA / SLEEP / STRESS / PRODUCTIVITY (jayjoshi37, n=11000)
# ==============================================================================
print(SEP); print("DATASET 2.1 - SOCIAL MEDIA SLEEP STRESS (jayjoshi37)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df1 = pd.read_csv(DATA + "2 Social Media Usage vs Sleep, Stress & Productivity/"
                         "social_media_sleep_stress_productivity_11000.csv")
print(f"Dimensiones: {df1.shape[0]} filas x {df1.shape[1]} columnas")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos totales:", df1.isna().sum().sum())

# PASO 3: Univariante --------------------------------------------------------
print("\nDistribucion stress_level:\n", df1["stress_level"].value_counts())
print(">>> Balance exacto ~33% por clase: primer indicio de datos sinteticos.")
print("\nDistribucion por plataforma:\n", df1["platform"].value_counts())

# PASO 4: Bivariante (codificacion ordinal del estres) -----------------------
df1["stress_ord"] = df1["stress_level"].map({"Low": 0, "Medium": 1, "High": 2})
num1 = df1.select_dtypes("number").drop(columns=["user_id"])
print("\nCorrelacion con stress_ord:\n",
      num1.corr()["stress_ord"].round(3).sort_values())
print("\nMedias por nivel de estres (sin diferencias entre grupos):")
print(df1.groupby("stress_level")[["social_media_hours", "sleep_hours",
      "productivity_score"]].mean().round(2))

# Figura 2.1
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.countplot(data=df1, x="stress_level", order=["Low", "Medium", "High"],
              ax=axes[0], color="#1F5C99")
axes[0].set_title(f"stress_level perfectamente balanceado (n={len(df1)})\nindicio de datos sintéticos")
sns.heatmap(num1.corr(), annot=True, fmt=".2f", cmap="vlag", center=0,
            ax=axes[1], annot_kws={"size": 8})
axes[1].set_title("Correlaciones ≈ 0 en todas las variables")
plt.tight_layout()
plt.savefig(FIG + "b2_1_social_media_sleep.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 2.1:
 - Todas las correlaciones con el estres son <=0.02: sin senal real.
 - Distribuciones balanceadas artificialmente (clases y plataformas).
 - CONCLUSION: dataset sintetico. Uso: validacion tecnica de pipeline y
   dashboards de volumen, no conclusiones sustantivas.
""")

# ==============================================================================
# DATASET 2.2 — SOCIAL MEDIA AND MENTAL HEALTH / SMMH (souvikahmed071, n=481)
# ==============================================================================
print(SEP); print("DATASET 2.2 - SMMH (souvikahmed071)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df2 = pd.read_csv(DATA + "2 Social Media and Mental Health/smmh.csv")
cols = list(df2.columns)
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")
print("Encuesta con escalas Likert 1-5 sobre uso de redes y bienestar.")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos totales:", df2.isna().sum().sum(),
      "(30 en columna de organizacion, no afecta el analisis)")

# PASO 3: Univariante --------------------------------------------------------
target2 = cols[18]   # "how often do you feel depressed or down?"
print(f"\nVariable objetivo: '{target2}'")
print(df2[target2].value_counts().sort_index())
print("\nTiempo diario en redes:\n", df2[cols[8]].value_counts())

# PASO 4: Bivariante ---------------------------------------------------------
likert_idx = [9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20]
likert_cols = [cols[i] for i in likert_idx]
num2 = df2[likert_cols].apply(pd.to_numeric, errors="coerce")
corr2 = num2.corr()[target2].round(2).sort_values(ascending=False)
print("\nCorrelaciones con sentirse deprimido/a:\n", corr2)

# Figura 2.2 (etiquetas cortas para legibilidad)
short = {
    cols[9]:  "Uso sin propósito",
    cols[10]: "Distracción por redes",
    cols[11]: "Inquietud sin redes",
    cols[12]: "Distracción fácil",
    cols[13]: "Preocupación excesiva",
    cols[14]: "Dificultad concentración",
    cols[15]: "Comparación social",
    cols[17]: "Búsqueda de validación",
    cols[19]: "Fluctuación de interés",
    cols[20]: "Problemas de sueño",
}
plot_data = corr2.drop(target2).rename(index=short)
plt.figure(figsize=(8, 5))
plot_data.sort_values().plot(kind="barh", color="#C44E52")
plt.title(f"Correlación con «sentirse deprimido/a» (n={len(df2)})\nSMMH — souvikahmed071")
plt.xlabel("Pearson r")
plt.tight_layout()
plt.savefig(FIG + "b2_2_smmh_corr.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 2.2:
 - Preocupacion excesiva: r=0.59 con sentirse deprimido/a (mayor del bloque).
 - Dificultad de concentracion: r=0.51; fluctuacion de interes: r=0.49.
 - Comparacion social: r=0.42; problemas de sueno: r=0.37.
 - CONCLUSION: no es el tiempo total en redes sino el patron de uso ansioso
   y comparativo lo que se asocia al malestar. Dataset mas valioso del bloque.
""")

# ==============================================================================
# DATASET 2.3 — MENTAL HEALTH LIFESTYLE (datafit17, n=3000)
# ==============================================================================
print(SEP); print("DATASET 2.3 - MENTAL HEALTH LIFESTYLE (datafit17)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df3 = pd.read_csv(DATA + "2 Mental_Health_Lifestyle GITHUB/"
                         "Mental_Health_Lifestyle_Dataset.csv")
print(f"Dimensiones: {df3.shape[0]} filas x {df3.shape[1]} columnas")

# PASO 2: Calidad (recodificacion de NaN semanticos) --------------------------
n_nan = df3["Mental Health Condition"].isna().sum()
df3["Mental Health Condition"] = df3["Mental Health Condition"].fillna("Ninguna")
print(f"NaN en Mental Health Condition: {n_nan} ({n_nan/len(df3)*100:.1f}%) "
      "-> recodificados como 'Ninguna' (ausencia de diagnostico, no dato faltante)")

# PASO 3: Univariante --------------------------------------------------------
print("\nDistribucion Stress Level:\n", df3["Stress Level"].value_counts())
print(">>> De nuevo balance casi exacto: indicio de datos sinteticos.")

# PASO 4: Bivariante ---------------------------------------------------------
df3["stress_ord"] = df3["Stress Level"].map({"Low": 0, "Moderate": 1, "High": 2})
num3 = df3.select_dtypes("number")
print("\nCorrelacion con stress_ord:\n",
      num3.corr()["stress_ord"].round(3).sort_values())

# Figura 2.3
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
orden = df3["Mental Health Condition"].value_counts().index
sns.countplot(data=df3, y="Mental Health Condition", order=orden,
              ax=axes[0], color="#8172B2")
axes[0].set_title(f"Condición de salud mental (n={len(df3)})")
sns.boxplot(data=df3, x="Stress Level", y="Sleep Hours",
            order=["Low", "Moderate", "High"], ax=axes[1],
            hue="Stress Level", palette="Blues", legend=False)
axes[1].set_title("Horas de sueño por nivel de estrés\n(sin diferencias entre grupos)")
plt.tight_layout()
plt.savefig(FIG + "b2_3_mental_health_lifestyle.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 2.3:
 - Correlaciones ~0.00-0.02 entre estres y todas las variables de estilo de vida.
 - Distribucion de estres artificialmente balanceada (1008/990/1002).
 - CONCLUSION: dataset sintetico. Mismo tratamiento que 2.1: valor tecnico
   (pipeline, dashboards multi-pais), no valor inferencial.
""")

print(SEP)
print("EDA BLOQUE 2 COMPLETADO - 3 figuras generadas en", FIG)
print(SEP)

DATASET 2.1 - SOCIAL MEDIA SLEEP STRESS (jayjoshi37)
Dimensiones: 11000 filas x 10 columnas
Nulos totales: 0

Distribucion stress_level:
 stress_level
Low       3715
Medium    3681
High      3604
Name: count, dtype: int64
>>> Balance exacto ~33% por clase: primer indicio de datos sinteticos.

Distribucion por plataforma:
 platform
LinkedIn     2239
Snapchat     2235
YouTube      2200
Twitter      2183
Instagram    2143
Name: count, dtype: int64

Correlacion con stress_ord:
 age                       -0.021
exercise_minutes          -0.012
daily_screen_time_hours   -0.010
social_media_hours        -0.008
productivity_score        -0.004
sleep_hours                0.000
study_work_hours           0.005
stress_ord                 1.000
Name: stress_ord, dtype: float64

Medias por nivel de estres (sin diferencias entre grupos):
              social_media_hours  sleep_hours  productivity_score
stress_level                                                     
High                        4.21

In [3]:
# -*- coding: utf-8 -*-
"""
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 3: PERCEPCION EMOCIONAL (IA / NLP)
================================================================================
Datasets analizados (2):
  3.1 dreaddit-train.csv + dreaddit-test.csv     (Kaggle: payaldhokane)
      -> corpus textual real de Reddit con etiqueta de estres. Fuente NLP.
  3.2 Depression+Student+Dataset.csv             (GitHub: prajwalg26)
      -> tabular; complementa el perfil de malestar estudiantil. NO es NLP.

Para el corpus textual, el EDA incluye pasos especificos de texto:
  PASO 3-T: longitud de posts, balance de clases, distribucion por subreddit
  PASO 4-T: senal lexica (sentimiento, categorias LIWC) por clase
================================================================================
"""
DATA = os.path.join(BASE, "Bloque 3") + os.sep

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

# ==============================================================================
# DATASET 3.1 — DREADDIT (payaldhokane, train n=2838 / test n=715)
# ==============================================================================
print(SEP); print("DATASET 3.1 - DREADDIT (payaldhokane)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
tr = pd.read_csv(DATA + "3 Stress Analysis in Social Media Dataset/dreaddit-train.csv")
te = pd.read_csv(DATA + "3 Stress Analysis in Social Media Dataset/dreaddit-test.csv")
print(f"Train: {tr.shape} | Test: {te.shape}")
print("Columnas clave: text (post), label (0/1), subreddit, sentiment, "
      "107 categorias LIWC, indices DAL y legibilidad (116 columnas total)")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos train:", tr.isna().sum().sum(), "| Nulos test:", te.isna().sum().sum())
print("Particion train/test ya definida por los autores: se respetara en el "
      "Objetivo 2 para comparabilidad con la literatura.")

# PASO 3-T: EDA textual - balance, longitud, subreddits ------------------------
tr["text_len"] = tr["text"].str.len()
tr["n_words"] = tr["text"].str.split().str.len()
print("\nBalance de clases:\n", tr["label"].value_counts())
print("\nLongitud (palabras) por clase:\n",
      tr.groupby("label")["n_words"].describe()[["mean", "50%", "max"]].round(1))
print("\nPosts por subreddit:\n", tr["subreddit"].value_counts())

# PASO 4-T: Senal lexica por clase --------------------------------------------
print("\nSentimiento medio por clase:\n",
      tr.groupby("label")["sentiment"].mean().round(3))
liwc_vars = ["lex_liwc_anx", "lex_liwc_negemo", "lex_liwc_posemo", "lex_liwc_i"]
print("\nCategorias LIWC por clase (medias):")
print(tr.groupby("label")[liwc_vars].mean().round(2))

# Figura 3.1a - balance, sentimiento, LIWC ansiedad
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.countplot(data=tr, x="label", ax=axes[0], color="#937860")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["No estrés (0)", "Estrés (1)"])
axes[0].set_title(f"Balance de clases (train, n={len(tr)})")
sns.boxplot(data=tr, x="label", y="sentiment", ax=axes[1],
            hue="label", palette=["#55A868", "#C44E52"], legend=False)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["No estrés", "Estrés"])
axes[1].set_title("Sentimiento por clase")
sns.barplot(data=tr, x="label", y="lex_liwc_anx", ax=axes[2],
            hue="label", palette=["#55A868", "#C44E52"], legend=False)
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(["No estrés", "Estrés"])
axes[2].set_title("Densidad de palabras de ansiedad (LIWC)\n2.5× mayor en la clase estrés")
plt.tight_layout()
plt.savefig(FIG + "b3_1_dreaddit_senales.png", dpi=130)
plt.close()

# Figura 3.1b - longitud y subreddits
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(data=tr, x="n_words", hue="label", bins=40, ax=axes[0],
             palette=["#55A868", "#C44E52"])
axes[0].set_title("Longitud de los posts (palabras) por clase")
axes[0].set_xlabel("Número de palabras")
tr["subreddit"].value_counts().sort_values().plot(kind="barh", ax=axes[1], color="#64B5CD")
axes[1].set_title("Posts por subreddit de origen")
plt.tight_layout()
plt.savefig(FIG + "b3_1_dreaddit_texto.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 3.1:
 - Clases casi balanceadas (1488 estres / 1350 no-estres): optimo para clasificar.
 - Sentimiento: +0.103 (no estres) vs -0.016 (estres).
 - Palabras de ansiedad LIWC: 1.28 vs 0.51 (x2.5 en clase estres).
 - Emocion negativa LIWC tambien discrimina; emocion positiva es mayor en no-estres.
 - Subreddits dominantes: ptsd, relationships, anxiety -> el estres capturado es
   relacional/clinico, no exclusivamente academico (considerar al generalizar).
 - 116 features precalculadas permiten comparar en el Objetivo 2 un modelo LIWC
   vs un modelo TF-IDF/embeddings propio.
""")

# ==============================================================================
# DATASET 3.2 — DEPRESSION STUDENT DATASET (prajwalg26, n=502)
# ==============================================================================
print(SEP); print("DATASET 3.2 - DEPRESSION STUDENT (prajwalg26)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df2 = pd.read_csv(DATA + "3 Student_Mental_Health_Analysis GITHUB/"
                         "Depression+Student+Dataset.csv")
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")
print("NOTA METODOLOGICA: dataset tabular (sin texto). Complementa el bloque con "
      "un perfil estructurado del malestar estudiantil; el NLP recae en Dreaddit.")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos totales:", df2.isna().sum().sum(),
      "| Duplicados:", df2.duplicated().sum())

# PASO 3: Univariante --------------------------------------------------------
print("\nBalance del target Depression:\n", df2["Depression"].value_counts())

# PASO 4: Bivariante ---------------------------------------------------------
ct = pd.crosstab(df2["Have you ever had suicidal thoughts ?"],
                 df2["Depression"], normalize="index") * 100
print("\n% Depresion segun pensamientos suicidas:\n", ct.round(1))
print("\nMedias por grupo de depresion:")
print(df2.groupby("Depression")[["Academic Pressure", "Study Satisfaction",
      "Financial Stress", "Study Hours"]].mean().round(2))

# Figura 3.2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ct["Yes"].plot(kind="bar", ax=axes[0], color=["#55A868", "#C44E52"])
axes[0].set_title("% con depresión según pensamientos suicidas")
axes[0].set_xlabel("¿Ha tenido pensamientos suicidas?")
axes[0].set_ylabel("% con depresión"); axes[0].tick_params(axis="x", rotation=0)
sns.boxplot(data=df2, x="Depression", y="Academic Pressure", ax=axes[1],
            hue="Depression", palette=["#55A868", "#C44E52"], legend=False)
axes[1].set_title("Presión académica según depresión")
plt.tight_layout()
plt.savefig(FIG + "b3_2_depression_student.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 3.2:
 - 73% de quienes reportan pensamientos suicidas presentan depresion (vs 26%).
 - Presion academica: 3.66 (con depresion) vs 2.34 (sin) -> +57%.
 - Estres financiero tambien es mayor en el grupo con depresion.
 - Target balanceado (252/250): apto para clasificacion en el Objetivo 2.
""")

print(SEP)
print("EDA BLOQUE 3 COMPLETADO - 3 figuras generadas en", FIG)
print(SEP)

DATASET 3.1 - DREADDIT (payaldhokane)
Train: (2838, 116) | Test: (715, 116)
Columnas clave: text (post), label (0/1), subreddit, sentiment, 107 categorias LIWC, indices DAL y legibilidad (116 columnas total)
Nulos train: 0 | Nulos test: 0
Particion train/test ya definida por los autores: se respetara en el Objetivo 2 para comparabilidad con la literatura.

Balance de clases:
 label
1    1488
0    1350
Name: count, dtype: int64

Longitud (palabras) por clase:
 

C:\Users\nicop\AppData\Local\Temp\ipykernel_8980\3604628787.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tr["text_len"] = tr["text"].str.len()
C:\Users\nicop\AppData\Local\Temp\ipykernel_8980\3604628787.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tr["n_words"] = tr["text"].str.split().str.len()


       mean   50%    max
label                   
0      82.2  78.0  255.0
1      88.8  83.0  310.0

Posts por subreddit:
 subreddit
ptsd                584
relationships       552
anxiety             503
domesticviolence    316
assistance          289
survivorsofabuse    245
homeless            168
almosthomeless       80
stress               64
food_pantry          37
Name: count, dtype: int64

Sentimiento medio por clase:
 label
0    0.103
1   -0.016
Name: sentiment, dtype: float64

Categorias LIWC por clase (medias):
       lex_liwc_anx  lex_liwc_negemo  lex_liwc_posemo  lex_liwc_i
label                                                            
0              0.51             2.07             3.42        7.11
1              1.28             4.42             2.04       10.84

HALLAZGOS 3.1:
 - Clases casi balanceadas (1488 estres / 1350 no-estres): optimo para clasificar.
 - Sentimiento: +0.103 (no estres) vs -0.016 (estres).
 - Palabras de ansiedad LIWC: 1.28 vs 0.51 (x2.5 en cla

In [4]:
# -*- coding: utf-8 -*-
"""
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 4: VALIDACION PSICOLOGICA
================================================================================
Datasets analizados (2):
  4.1 data.csv (DASS-42)                    (Kaggle: lucasgreenwell)
      + codebook oficial: "Depression Anxiety Stress Scales Responses.txt"
  4.2 Mental Health dataset1.csv            (Kaggle: chanchal24 / input)

El DASS-42 es una escala clinicamente validada. La clave de subescalas y los
codigos demograficos provienen del codebook oficial incluido en el ZIP.
================================================================================
"""
DATA = os.path.join(BASE, "Bloque 4") + os.sep

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

# ==============================================================================
# DATASET 4.1 — DASS-42 (lucasgreenwell, n=39.775)
# ==============================================================================
print(SEP); print("DATASET 4.1 - DASS-42 (lucasgreenwell)"); print(SEP)

# PASO 1: Carga (separador TAB, no coma) --------------------------------------
df = pd.read_csv(DATA + "4 Depression Anxiety Stress Scales Responses/data.csv",
                 sep="\t")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print("42 items (respuesta A, tiempo E, posicion I) + TIPI + demograficos")

# PASO 2: Calidad - limpieza de edades invalidas -------------------------------
# El codebook confirma que 'age' es respuesta libre: hay valores tipo 1998 (ano)
n_before = len(df)
df = df[(df["age"] >= 13) & (df["age"] <= 90)].copy()
print(f"Filtrado de edad a rango 13-90: {n_before} -> {len(df)} "
      f"({n_before - len(df)} registros eliminados)")

# PASO 3: Calculo de subescalas (clave oficial DASS-42 del codebook) ----------
dep_items = [3, 5, 10, 13, 16, 17, 21, 24, 26, 31, 34, 37, 38, 42]
anx_items = [2, 4, 7, 9, 15, 19, 20, 23, 25, 28, 30, 36, 40, 41]
str_items = [1, 6, 8, 11, 12, 14, 18, 22, 27, 29, 32, 33, 35, 39]

def subescala(items):
    """Suma las respuestas (QnA) de los items de una subescala. Rango 14-56."""
    return df[[f"Q{i}A" for i in items]].sum(axis=1)

df["dep_score"] = subescala(dep_items)
df["anx_score"] = subescala(anx_items)
df["str_score"] = subescala(str_items)

# Subconjunto proxy de poblacion estudiantil: 18-25 anos
stu = df[(df["age"] >= 18) & (df["age"] <= 25)].copy()
print(f"\nSubconjunto proxy estudiantil (18-25 anos): n={len(stu)}")
print("\nEstadisticos de las subescalas (subset 18-25):")
print(stu[["dep_score", "anx_score", "str_score"]].describe().round(1).loc[
      ["mean", "std", "50%"]])

# PASO 4: Desagregacion demografica (codigos del codebook oficial) ------------
stu["genero"] = stu["gender"].map({1: "Hombre", 2: "Mujer", 3: "Otro"})
stu["educacion"] = stu["education"].map({1: "Menos que secundaria",
                                          2: "Secundaria",
                                          3: "Grado universitario",
                                          4: "Posgrado"})
print("\nSubescalas medias por genero:\n",
      stu.groupby("genero")[["dep_score", "anx_score", "str_score"]].mean().round(1))
print("\nSubescalas medias por nivel educativo:\n",
      stu.groupby("educacion")[["dep_score", "anx_score", "str_score"]].mean().round(1))

# Figura 4.1a - distribuciones de las 3 subescalas
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
nombres = {"dep_score": "Depresión", "anx_score": "Ansiedad", "str_score": "Estrés"}
for ax, (col, name) in zip(axes, nombres.items()):
    sns.histplot(stu[col], bins=21, ax=ax, color="#1F5C99")
    ax.axvline(stu[col].mean(), color="#C44E52", ls="--", lw=1.5,
               label=f"media={stu[col].mean():.1f}")
    ax.set_title(f"DASS — {name}"); ax.set_xlabel("Puntuación (14-56)")
    ax.legend()
fig.suptitle(f"Subescalas DASS-42, subset 18-25 años (n={len(stu)})", y=1.03)
plt.tight_layout()
plt.savefig(FIG + "b4_1_dass_subescalas.png", dpi=130, bbox_inches="tight")
plt.close()

# Figura 4.1b - desagregacion demografica
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
stu.groupby("genero")[["dep_score", "anx_score", "str_score"]].mean() \
   .rename(columns=nombres) \
   .plot(kind="bar", ax=axes[0], color=["#1F5C99", "#DD8452", "#55A868"])
axes[0].set_title("DASS medio por género"); axes[0].tick_params(axis="x", rotation=0)
axes[0].set_ylabel("Puntuación media")
orden_edu = ["Menos que secundaria", "Secundaria", "Grado universitario", "Posgrado"]
stu.groupby("educacion")[["dep_score", "anx_score", "str_score"]].mean() \
   .reindex(orden_edu).rename(columns=nombres) \
   .plot(kind="bar", ax=axes[1], color=["#1F5C99", "#DD8452", "#55A868"])
axes[1].set_title("DASS medio por nivel educativo")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(FIG + "b4_1_dass_demograficos.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 4.1:
 - Subset 18-25 (n=23.031): medias Dep=35.0, Anx=30.4, Str=35.2 (escala 14-56).
 - Mujeres reportan mas ansiedad (31.0) que hombres (28.1); el grupo 'Otro'
   presenta el mayor malestar en las tres subescalas.
 - A menor nivel educativo completado, mayor puntuacion en las 3 subescalas.
 - Referencia normativa robusta para contrastar los niveles de estres
   autoreportados en los Bloques 1 y 2 (sin fusionar datos).
""")

# ==============================================================================
# DATASET 4.2 — MENTAL HEALTH DATASET (chanchal24, n=261.328)
# ==============================================================================
print(SEP); print("DATASET 4.2 - MENTAL HEALTH DATASET (chanchal24)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df2 = pd.read_csv(DATA + "4 Mental Health Dataset/Mental Health dataset1.csv")
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")

# PASO 2: Calidad - columna duplicada y nulos ---------------------------------
if (df2["SocialWeakness"] == df2["SocialWeakness.1"]).all():
    df2 = df2.drop(columns=["SocialWeakness.1"])
    print("Columna 'SocialWeakness.1' eliminada (duplicado exacto).")
print("Nulos:", dict(df2.isna().sum()[df2.isna().sum() > 0]))

# PASO 3: Filtro a poblacion estudiantil --------------------------------------
stu2 = df2[df2["Occupation"] == "Student"].copy()
print(f"\nSubconjunto Occupation=='Student': n={len(stu2)}")
print("\nDistribucion IncreasingStress (estudiantes):\n",
      stu2["IncreasingStress"].value_counts())

# PASO 4: Bivariante - estres creciente vs busqueda de tratamiento ------------
ct = pd.crosstab(stu2["IncreasingStress"], stu2["Treatment"],
                 normalize="index") * 100
print("\n% que busca tratamiento segun estres creciente:\n", ct.round(1))
ct2 = pd.crosstab(stu2["FamilyHistory"], stu2["Treatment"], normalize="index") * 100
print("\n% que busca tratamiento segun antecedentes familiares:\n", ct2.round(1))

# Figura 4.2
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ct.plot(kind="bar", stacked=True, ax=axes[0], color=["#CCCCCC", "#1F5C99"])
axes[0].set_title(f"Tratamiento según estrés creciente\n(estudiantes, n={len(stu2)})")
axes[0].set_ylabel("%"); axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Tratamiento", loc="lower right")
ct2.plot(kind="bar", stacked=True, ax=axes[1], color=["#CCCCCC", "#C44E52"])
axes[1].set_title("Tratamiento según antecedentes familiares")
axes[1].set_ylabel("%"); axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Tratamiento", loc="lower right")
plt.tight_layout()
plt.savefig(FIG + "b4_2_chanchal_tratamiento.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 4.2:
 - La busqueda de tratamiento apenas varia con el estres creciente (48-51%):
   posible brecha de acceso o estigma en poblacion estudiantil.
 - Los antecedentes familiares SI elevan la busqueda de tratamiento.
 - Dataset masivo (54.207 estudiantes): apto para segmentaciones en dashboards.
""")

print(SEP)
print("EDA BLOQUE 4 COMPLETADO - 3 figuras generadas en", FIG)
print(SEP)

DATASET 4.1 - DASS-42 (lucasgreenwell)
Dimensiones: 39775 filas x 172 columnas
42 items (respuesta A, tiempo E, posicion I) + TIPI + demograficos
Filtrado de edad a rango 13-90: 39775 -> 39767 (8 registros eliminados)

Subconjunto proxy estudiantil (18-25 anos): n=23039

Estadisticos de las subescalas (subset 18-25):
      dep_score  anx_score  str_score
mean       35.0       30.4       35.2
std        12.0        9.9       10.3
50%        35.0       29.0       35.0

Subescalas medias por genero:
         dep_score  anx_score  str_score
genero                                 
Hombre       34.4       28.1       32.8
Mujer        35.1       31.0       35.7
Otro         40.2       33.0       37.8

Subescalas medias por nivel educativo:
                       dep_score  anx_score  str_score
educacion                                            
Grado universitario        34.2       30.1       34.8
Menos que secundaria       39.1       32.9       37.5
Posgrado                   34.1       29